In [ ]:
# Kaggle训练脚本
%config InlineBackend.figure_format = 'retina'
import os
from pathlib import Path
from dotenv import load_dotenv

# 1. 加载环境变量
env_path = Path('/kaggle/input/env-key/.env')

if env_path.exists():
    load_dotenv(dotenv_path=env_path, override=True)
    print("✅ 环境变量已从数据集成功加载")
else:
    print("❌ 未找到 .env 文件，请检查数据集挂载路径")

# 验证加载结果
MODELSCOPE_TOKEN = os.getenv("MODELSCOPE_TOKEN")

!echo $MODELSCOPE_TOKEN
!nvidia-smi

# 2. 设置目录结构
WORK_DIR = Path("/tmp/train-llm")  # 代码与环境（不保存）
ARTIFACTS_DIR = Path("/kaggle/working/artifacts")  # 持久化目录（保存）

# 设置环境变量，让所有脚本都使用这个artifacts目录
os.environ['ARTIFACTS_DIR'] = str(ARTIFACTS_DIR)

print(f"📁 工作目录（临时）: {WORK_DIR}")
print(f"📁 Artifacts目录（持久化）: {ARTIFACTS_DIR}")

# 3. 克隆代码到临时目录
!cd /tmp && rm -rf train-llm && git clone https://github.com/try-agaaain/train-llm.git

# 4. 安装依赖到临时目录
!pip install uv
!cd {WORK_DIR} && uv sync --index-url https://pypi.org/simple

# 5. 运行评估/数据拉取（由Makefile统一路径与备份策略处理）
!cd {WORK_DIR} && make dpull MODELSCOPE_TOKEN=$MODELSCOPE_TOKEN ARTIFACTS_DIR={ARTIFACTS_DIR}
!cd {WORK_DIR} && make mpull MODELSCOPE_TOKEN=$MODELSCOPE_TOKEN ARTIFACTS_DIR={ARTIFACTS_DIR}
!cd {WORK_DIR} && make evaluate EVAL_BATCH_SIZE=8 ARTIFACTS_DIR={ARTIFACTS_DIR}

# 6. 推送数据集到ModelScope
!cd {WORK_DIR} && MODELSCOPE_TOKEN=$MODELSCOPE_TOKEN make dpush ARTIFACTS_DIR={ARTIFACTS_DIR}

print("\n✅ 训练完成！")
print(f"📦 Artifacts已保存在: {ARTIFACTS_DIR}")

✅ 环境变量已从数据集成功加载
ms-3e3af35b-ba13-4c77-9728-526f1325e8d6
Tue Jan  6 15:20:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                